# In Vitro Data Duplicate Analysis

Check duplicated rows and duplicated SMILES structures in each file.

In [3]:
import pandas as pd
import os
from collections import defaultdict

data_dir = r'../data/in_vitro_data'
files = [
    'chembl_CL_raw.csv',
    'chembl_Fu_raw.csv',
    'biogen_CL_raw.csv',
    'chembl_PPB_raw.csv',
    'tdc_cl_hep.csv',
    'tdc_cl_mic.csv',
    'tdc_half_life.csv',
    'tdc_PPBR_AZ.csv'
]

# Load each file
dataframes = {}
for file in files:
    file_path = os.path.join(data_dir, file)
    try:
        df = pd.read_csv(file_path)
        dataframes[file] = df
        print(f"✓ {file}: {len(df)} rows, {len(df.columns)} columns")
    except Exception as e:
        print(f"✗ {file}: Error - {e}")

✓ chembl_CL_raw.csv: 80447 rows, 8 columns
✓ chembl_Fu_raw.csv: 14033 rows, 8 columns
✓ biogen_CL_raw.csv: 3521 rows, 10 columns
✓ chembl_PPB_raw.csv: 13778 rows, 8 columns
✓ tdc_cl_hep.csv: 1213 rows, 3 columns
✓ tdc_cl_mic.csv: 1102 rows, 3 columns
✓ tdc_half_life.csv: 667 rows, 3 columns
✓ tdc_PPBR_AZ.csv: 1614 rows, 3 columns


## 1. SMILES-based duplicate check (duplicate molecular structures)

In [5]:
print("\n" + "=" * 80)
print("SMILES-based duplicate check")
print("=" * 80)

smiles_column_candidates = ['smiles', 'SMILES', 'Smiles']

for file, df in dataframes.items():
    # Find the SMILES column
    smiles_col = None
    for col in smiles_column_candidates:
        if col in df.columns:
            smiles_col = col
            break
    
    if smiles_col:
        duplicated_smiles = df[smiles_col].duplicated(keep=False).sum()
        unique_smiles = df[smiles_col].nunique()
        total_rows = len(df)
        
        print(f"\n📌 {file}")
        print(f"   Total rows: {total_rows}")
        print(f"   Unique SMILES: {unique_smiles}")
        print(f"   Rows with duplicated SMILES: {duplicated_smiles}")
        
        if duplicated_smiles > 0:
            dup_mask = df[smiles_col].duplicated(keep=False)
            dup_smiles = df[dup_mask].sort_values(by=smiles_col)
            print(f"\n   Duplicated SMILES (top 5):")
            for smiles_val in dup_smiles[smiles_col].unique()[:5]:
                count = (df[smiles_col] == smiles_val).sum()
                print(f"      - {smiles_val[:50]}... ({count} occurrences)")
    else:
        print(f"✗ {file}: no SMILES column")


SMILES-based duplicate check

📌 chembl_CL_raw.csv
   Total rows: 80447
   Unique SMILES: 44278
   Rows with duplicated SMILES: 54312

   Duplicated SMILES (top 5):
      - Br.COc1ccc2c3c1O[C@H]1C[C@@H](O)C=C[C@@]31CCN(C)C2... (3 occurrences)
      - Br.Cc1cnc(Nc2ccc(C#N)cc2)nc1Nc1c(F)cc(-c2ccncc2)cc... (4 occurrences)
      - Br.NC(=O)c1cnc2[nH]ccc2c1NC1C2CC3CC1CC(O)(C3)C2... (2 occurrences)
      - Br.O=C1NCC(c2ccccc2)C12CCN(C1(c3ccccc3)CCCCC1)CC2... (2 occurrences)
      - Brc1cc(Br)cc([C@H]2C[C@@H](CN3CCCC3)N2Cc2ccccc2Br)... (2 occurrences)

📌 chembl_Fu_raw.csv
   Total rows: 14033
   Unique SMILES: 7248
   Rows with duplicated SMILES: 10070

   Duplicated SMILES (top 5):
      - Br.CN(C)CCCC1(c2ccc(F)cc2)OCc2cc(C#N)ccc21... (4 occurrences)
      - Brc1ccc2[nH]nc(C3=CCCNC3)c2c1... (3 occurrences)
      - C#CCCCC(=O)c1cc(C(C)(C)C)c(O)c(C(C)(C)C)c1... (3 occurrences)
      - C#CCN(C)[C@H](C)Cc1ccccc1... (2 occurrences)
      - C#CCN(C)[C@H](C[18F])Cc1ccccc1... (2 occurrences)

📌 biog

## 3. Cross-file duplicate check (is the same molecule in several files?)

In [6]:
print("\n" + "=" * 80)
print("Cross-file SMILES duplicate check")
print("=" * 80)

# Collect the SMILES of each file
file_smiles = {}
for file, df in dataframes.items():
    smiles_col = None
    for col in smiles_column_candidates:
        if col in df.columns:
            smiles_col = col
            break
    
    if smiles_col:
        file_smiles[file] = set(df[smiles_col].dropna().astype(str))

# Check overlap between file pairs
files_with_smiles = list(file_smiles.keys())
for i, file1 in enumerate(files_with_smiles):
    for file2 in files_with_smiles[i+1:]:
        overlap = file_smiles[file1] & file_smiles[file2]
        if len(overlap) > 0:
            print(f"\n📌 {file1} ↔ {file2}")
            print(f"   Shared SMILES: {len(overlap)}")
            print(f"   Examples (top 3):")
            for smiles in list(overlap)[:3]:
                print(f"      - {smiles[:60]}...")


Cross-file SMILES duplicate check

📌 chembl_CL_raw.csv ↔ chembl_Fu_raw.csv
   Shared SMILES: 4585
   Examples (top 3):
      - CC(=O)O[C@H]1C(=O)[C@@]2(C)[C@H]([C@H](OC(=O)c3ccccc3)[C@]3(...
      - N#Cc1ccc2ccc(=O)n(CCN3CC[C@H](NCc4cc5c(cn4)OCCO5)[C@H](F)C3)...
      - COc1ccc2ncc(F)c(CCN3CCC(NCc4cc5c(cn4)OCCO5)CC3)c2n1...

📌 chembl_CL_raw.csv ↔ biogen_CL_raw.csv
   Shared SMILES: 184
   Examples (top 3):
      - C=CC(=O)N1CCC[C@@H](n2nc(-c3cccc(C(=O)Nc4ccc(C(C)C)c(C)c4)c3...
      - CNC(=O)c1ccccc1Nc1nc(Nc2ccc(N3CCOCC3)cc2OC)ncc1Cl...
      - C=CCc1ccccc1OCC(O)CNC(C)C...

📌 chembl_CL_raw.csv ↔ chembl_PPB_raw.csv
   Shared SMILES: 4625
   Examples (top 3):
      - CC(C)CCCCC(=O)N[C@@H](CCN)C(=O)N[C@H](C(=O)N[C@@H](CCN)C(=O)...
      - COc1cc(N[C@H](C(=O)c2c[nH]c3ccc(OC(F)(F)F)cc23)c2ccc(Cl)cc2O...
      - COc1ccc2ncc(F)c(CCN3CCC(NCc4cc5c(cn4)OCCO5)CC3)c2n1...

📌 chembl_Fu_raw.csv ↔ biogen_CL_raw.csv
   Shared SMILES: 68
   Examples (top 3):
      - CS(=O)(=O)c1ccc(-c2cnc(NCc3ccco3)n3

## 4. Summary

In [7]:
print("\n" + "=" * 80)
print("Summary")
print("=" * 80)

total_rows_all_files = sum(len(df) for df in dataframes.values())
total_files = len(dataframes)

print(f"\nTotal files: {total_files}")
print(f"Total rows (all files combined): {total_rows_all_files}")

# Basic statistics per file
print(f"\nBasic statistics per file:")
for file, df in sorted(dataframes.items()):
    print(f"  {file:<30} {len(df):>6} rows")


Summary

Total files: 8
Total rows (all files combined): 116375

Basic statistics per file:
  biogen_CL_raw.csv                3521 rows
  chembl_CL_raw.csv               80447 rows
  chembl_Fu_raw.csv               14033 rows
  chembl_PPB_raw.csv              13778 rows
  tdc_PPBR_AZ.csv                  1614 rows
  tdc_cl_hep.csv                   1213 rows
  tdc_cl_mic.csv                   1102 rows
  tdc_half_life.csv                 667 rows
